# Pulling BRCA2 variants from the All of US V8 Whole Genome Sequencing Data and Combining Annotation Variables.

### Loading packages and creating cloud folder

In [1]:
## import packages
from datetime import datetime #usually used to check runtimes of code
import os #allows reading environment variables (ex: WORKSPACE_BUCKET).
import pandas as pd #used in this analysis to turn items into DFs for CSVs
from collections import Counter #Counts how many times items appear in a list or dataset
import hail.expr.aggregators as agg #large-scale genetic data summary calculations inside Hail (like AF, counts, etc.)
from pprint import pprint #prints cleaner formats
import numpy as np #usually for stats and arrays
import matplotlib.pyplot as plt #for plotting

Loading BokehJS ...

In [ ]:
#generates new cloud folder for this notebook's files
bucket = os.getenv('WORKSPACE_BUCKET')

In [3]:
import hail as hl #importing hail package, doesn't confirm installation till later step

In [4]:
## set up hail
hl.init(default_reference = "GRCh38") #confirming hail install and setting DNA reference structure to GRch38

/opt/conda/lib/python3.10/site-packages/hail/context.py:350: UserWarning:

Using hl.init with a default_reference argument is deprecated. To set a default reference genome after initializing hail, call `hl.default_reference` with an argument to set the default reference genome.

/opt/conda/lib/python3.10/site-packages/hailtop/aiocloud/aiogoogle/user_config.py:43: UserWarning:

Reading spark-defaults.conf to determine GCS requester pays configuration. This is deprecated. Please use `hailctl config set gcs_requester_pays/project` and `hailctl config set gcs_requester_pays/buckets`.

Running on Apache Spark version 3.5.3
SparkUI available at http://all-of-us-36156-m.us-central1-c.c.terra-vpc-sc-e1ba3f8a.internal:33785
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.134-952ae203dbbe
LOGGING: writing to /home/jupyter/workspaces/sandboxanalysesofbrca1andbrca2variantsfromwgsdatafromtheaou/hail-20260318-1427-0.2.134-952ae203dbbe.log
202

### importing data

In [5]:
#Pulling the V8 exome variant data as a matrix table from AOU CDR directory
#https://support.researchallofus.org/hc/en-us/articles/29475233432212-Controlled-CDR-Directory
mt_wgs_exome_path = "gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/splitMT/hail.mt"

#Setting up V8 Matrix Table with Exome counts 
mt = hl.read_matrix_table(mt_wgs_exome_path)
current_ct = mt.count()
print(f"The count for the current analysis is {current_ct}") #(45704594, 414830)

#(X, Y) variants count number vs number of participants

The count for the current analysis is (45704594, 414830)


In [6]:
mt.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'filters': set<str>
    'a_index': int32
    'was_split': bool
    'variant_qc': struct {
        gq_stats: struct {
            mean: float64, 
            stdev: float64, 
            min: float64, 
            max: float64
        }, 
        call_rate: float64, 
        n_called: int64, 
        n_not_called: int64, 
        n_filtered: int64, 
        n_het: int64, 
        n_non_ref: int64, 
        het_freq_hwe: float64, 
        p_value_hwe: float64, 
        p_value_excess_het: float64
    }
    'info': struct {
        AC: array<int32>, 
        AF: array<float64>, 
        AN: int32, 
        homozygote_count: array<int32>
    }
----------------------------------------
Entry fields:
    'GQ': int32
    'PS': int64
    'RGQ': int32
    

### Filter hail matrix to analyze only BRCA2: chr13:32310496-32405265

In [7]:
#Now setting the filter for BRCA2 variants
BRCA2intervals = ['chr13:32310496-32405265']

#filtering the BRCA2 variants from the overall MT
mtB2 = hl.filter_intervals(
    mt,
    [hl.parse_locus_interval(x,)
     for x in BRCA2intervals])

In [9]:
#count of variants, number of participants for BRCA2
currentB2_fct = mtB2.count() #(7981, 414830)
print(f"The count for the current B2 filtered analysis is {currentB2_fct}")

The count for the current B2 filtered analysis is (7981, 414830)


### Adding the ancestry annotation to the Matrix Table

In [10]:
#All of Us already did the predicted ancestry analysis based on WGS data for V8 in the CDR files
#We are naming the ancestry data as test_file
#this file contains the ancestry information of all participants collected in the WGS file

test_file = "gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv"

In [ ]:
#Read in test_file TSV and assign research_id as a hail string variable to combine with mt variable 's' 
#which is the sample id
ancestry = (hl.import_table(test_file,
                              types={'research_id':hl.tstr}, #making sure research_is is a string to later match the mt sample id which is also string  
                            impute=True, 
                              key='research_id'))

# This line "prints" the first 10 rows of the ancestry file
ancestry.show() #removed output to comply with All of Us Data User Policies

In [12]:
#confirming the unique values for the ancestry_pred column we use later
ancestry.aggregate(hl.agg.counter(ancestry.ancestry_pred))
#all catagories captured in our filter function in the last section

{'afr': 84148,
 'amr': 79106,
 'eas': 10099,
 'eur': 234353,
 'mid': 1545,
 'sas': 5579}

In [13]:
#This step adds the ancestry labels into the column fields of the MT, matching the sample/researcher IDs together
mtB2_AC = mtB2.annotate_cols(ap = ancestry[mtB2.s])

#Here we can see how the ancestry feilds are added. The columns that were in the ancestry tables are now MT col fields
mtB2_AC.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
    'ap': struct {
        ancestry_pred: str, 
        probabilities: str, 
        pca_features: str, 
        ancestry_pred_other: str
    }
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'filters': set<str>
    'a_index': int32
    'was_split': bool
    'variant_qc': struct {
        gq_stats: struct {
            mean: float64, 
            stdev: float64, 
            min: float64, 
            max: float64
        }, 
        call_rate: float64, 
        n_called: int64, 
        n_not_called: int64, 
        n_filtered: int64, 
        n_het: int64, 
        n_non_ref: int64, 
        het_freq_hwe: float64, 
        p_value_hwe: float64, 
        p_value_excess_het: float64
    }
    'info': struct {
        AC: array<int32>, 
        AF: array<float64>, 
        AN: int32, 
        h

### Building the VAT Table for BRCA2 V8

In [14]:
#Pulling the directory path from the CDR for the Variant Annonation Table file
#!gsutil -u $$GOOGLE_PROJECT ls -l gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/vat/vat_complete.bgz.tsv.gz
vat_path = 'gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/vat/vat_complete.bgz.tsv.gz'

#looking at the size of the file (1 objects, 159173688181 bytes (148.24 GiB))
!gsutil -u $$GOOGLE_PROJECT ls -l {vat_path}

#importing the table
vat_table = hl.import_table(vat_path, force=True, quote='"', delimiter="\t", force_bgz=True)

#filtering by gene symbol
gene='BRCA2'
vat_new = vat_table.filter(vat_table["gene_symbol"]==gene)

#filtering transcript by canonical
vat_new = vat_new.filter(vat_new["is_canonical_transcript"].lower()== "true")

#selecting the fields I need
vat_new=vat_new.select('vid','contig','position','consequence','clinvar_classification','variant_type','ref_allele','alt_allele',
                       'gvs_all_ac','gvs_all_an','gvs_all_af','gvs_all_sc','dbsnp_rsid','revel','gvs_max_subpop','aa_change',
                      'transcript','gene_symbol','gvs_max_ac','gvs_max_sc','gvs_max_an','gnomad_max_subpop','is_canonical_transcript')

#reviewing structure created
vat_new.describe()

159173688181  2025-02-14T20:57:44Z  gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/vat/vat_complete.bgz.tsv.gz
TOTAL: 1 objects, 159173688181 bytes (148.24 GiB)
----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    'vid': str 
    'contig': str 
    'position': str 
    'consequence': str 
    'clinvar_classification': str 
    'variant_type': str 
    'ref_allele': str 
    'alt_allele': str 
    'gvs_all_ac': str 
    'gvs_all_an': str 
    'gvs_all_af': str 
    'gvs_all_sc': str 
    'dbsnp_rsid': str 
    'revel': str 
    'gvs_max_subpop': str 
    'aa_change': str 
    'transcript': str 
    'gene_symbol': str 
    'gvs_max_ac': str 
    'gvs_max_sc': str 
    'gvs_max_an': str 
    'gnomad_max_subpop': str 
    'is_canonical_transcript': str 
----------------------------------------
Key: []
----------------------------------------


In [ ]:
#DO NOT RE-RUN YOU WILL OVERWRITE THE VAT FILE CREATED. IT TAKES 5.5 HOURS TO CREATE
#saving the VAT table to the cloud 
out_pathB1_3 = f'{bucket}/data/vat_gene_{gene}_v8_2026_hail_CT_GH.tsv'

In [16]:
#capturing the time taken to run the VAT Table
start = datetime.now()
print("This project started running at", start)

This project started running at 2026-03-18 14:37:00.626232


In [17]:
#DO NOT RE-RUN YOU WILL OVERWRITE THE VAT FILE CREATED. IT TAKES 5.5 HOURS TO CREATE
#save to the bucket, will take time, Takes approx. 5.5 hours
vat_new.export(out_pathB1_3)

In [18]:
#total time Vat Table ran
end = datetime.now()
print(end-start)

4:42:36.180697


In [ ]:
#Check saved file in the bucket
!gsutil ls -l {bucket}/data/vat_gene_{gene}_v8_2026_hail_CT.tsv
#output removed to comply with AOU data use policies
#it should show the cloud bucket file location and the size of the file

In [23]:
#Importing file straight back into a table, file name is fictious but right format
vat_filename = 'gs://fc-secure-123-456-789/data/vat_gene_BRCA2_v8_2026_hail_CT.tsv'
vat_table = hl.import_table(vat_filename,
                            impute = True,
                            )

In [21]:
#The number of BRCA2 Variants captured in the filtered VAT table (not intervaled) 43424
#this is all the BRCA2 variants in the VAT table, this includes BRCA2 outside the chormosome region we are interested in
vat_table.count()

43424

In [25]:
#checking the structure of the table to ensure it is the same information we saw before exporting
vat_table.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    'vid': str 
    'contig': str 
    'position': int32 
    'consequence': str 
    'clinvar_classification': str 
    'variant_type': str 
    'ref_allele': str 
    'alt_allele': str 
    'gvs_all_ac': int32 
    'gvs_all_an': int32 
    'gvs_all_af': float64 
    'gvs_all_sc': int32 
    'dbsnp_rsid': str 
    'revel': str 
    'gvs_max_subpop': str 
    'aa_change': str 
    'transcript': str 
    'gene_symbol': str 
    'gvs_max_ac': int32 
    'gvs_max_sc': int32 
    'gvs_max_an': int32 
    'gnomad_max_subpop': str 
    'is_canonical_transcript': bool 
----------------------------------------
Key: []
----------------------------------------


### Joining the VAT table with the MatrixTable

In [26]:
#Reformatting vid variable, to look like MatrixTable's variants to link genomic data when merging
#vid orignially looked like 1-3414320-G-A
vat_table = vat_table.annotate(vid_alt = hl.str('chr') + vat_table.vid.replace('-', ':'))
vat_table = vat_table.key_by(**hl.parse_variant(vat_table.vid_alt, reference_genome = 'GRCh38'))

In [27]:
#adding vat variables names to the BRCA2 mt for the later join
mtB2_26_CV = mtB2_AC.annotate_rows(vat = vat_table[mtB2_AC.row_key])

#joining BRCA2 mt with the vat table
mtB2_26_CV = mtB2_26_CV.semi_join_rows(vat_table)

#looking at merged metadata
mtB2_26_CV.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
    'ap': struct {
        ancestry_pred: str, 
        probabilities: str, 
        pca_features: str, 
        ancestry_pred_other: str
    }
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'filters': set<str>
    'a_index': int32
    'was_split': bool
    'variant_qc': struct {
        gq_stats: struct {
            mean: float64, 
            stdev: float64, 
            min: float64, 
            max: float64
        }, 
        call_rate: float64, 
        n_called: int64, 
        n_not_called: int64, 
        n_filtered: int64, 
        n_het: int64, 
        n_non_ref: int64, 
        het_freq_hwe: float64, 
        p_value_hwe: float64, 
        p_value_excess_het: float64
    }
    'info': struct {
        AC: array<int32>, 
        AF: array<float64>, 
        AN: int32, 
        h

In [28]:
#counting the variants after the join of the mt and vat (7629, 414830) we lost about 352 variants in the merge
#meaning VAT table might not have had these variants in it's table
mtB2_26_CV.count()

(7629, 414830)

### Get variants that passes the QC test separated by ancestry
#### To do this I will define two functions
#### 1) save_to_csv(mt, name):
#### 2) filter_by_ancestry(mt, ancestry)

In [29]:
#This function takes the MT and extracts the row/variant information to convert to a df. 
#Essentially putting the created mt_filtered qc rows(from other function) into its own dataframe and saving to our file
#named by the ancestry in the loop list 
def save_to_csv(mt, name):
    ''' mt is supposed to be already QC''' #QC was calculated for the whole unfiltered mt, must re-run QC when new data added
    ## This line will get the matrix and add the variants information to a panda dataframe
    dataframe = mt.rows().to_pandas()
    ### this line just take the dataframe and saveit on the bucket
    dataframe.to_csv(bucket + "/2026_B2_CT_variants_%s.csv"%(name), sep=" ", header=True, index=False)
    ## save on the local environment, will be deleted, but it's easier to download to my PC
    dataframe.to_csv("./2026_B2_CT_variants_%s.csv"%(name), sep=" ", header=True, index=False)

In [30]:
#This function filters the MatrixTable by ancestry input
#Then calculates quality metrics(variant_qc) for the filtered variants such as AF and call rate from the schema
#From there it filters based on AF and call rate 
def filter_by_ancestry(mt, ancestry):
    ''' This function will take the hail matrix already annotated by ancestry and filtered by BRCA location'''
    ''' It will filtered by ancestry and return the filtered matrix'''
    mt_filt = mt.filter_cols(mt.ap.ancestry_pred == ancestry)
    mt_filt = hl.variant_qc(mt_filt)
    #CG changed AF > 0.0 on 9-20-23
    mt_filt = mt_filt.filter_rows(mt_filt.variant_qc.AF[1] > 0.0, keep=True)
    mt_filt = mt_filt.filter_rows(mt_filt.variant_qc.call_rate > 0.90, keep=True)
    return mt_filt

In [31]:
#Now I just applied the functions to the different ancestries
#How functions are used: A for loop with the different ancestries are listed 
#Iteration goes through each ancestry, filters mt by the ancestry, creates qc stats for the filtered data, 
#filters the qc stats on AF and call rate, then saves the filtered qc stats as a individual csv with the name variants_ancestry 
for ancestry in ['afr', 'amr', 'eas', 'eur', 'mid', 'sas']:
    mt_filtered = filter_by_ancestry(mtB2_26_CV, ancestry)
    save_to_csv(mt_filtered, ancestry)

In [ ]:
#lists files in our cloud bucket
!gsutil -u $GOOGLE_PROJECT ls -lh gs://fc-secure-19c6c9d4-9ef0-4b1b-9492-700ceb5f739e
#files removed to comply with AOU data user policies. But should show a list of file locations 
#including the ones we created in the steps above

In [33]:
#loads in the afr ancestry file we just created
#amr = pd.read_csv(bucket + "/variants_amr.csv", delim_whitespace=True)
amr = pd.read_csv('gs://fc-secure-123-456-789/2026_B2_CT_variants_amr.csv', delim_whitespace=True) #file ficitious

In [ ]:
'''
The notebook runs completely, the results should be stored in variants_*.csv files. 
https://support.researchallofus.org/hc/en-us/articles/4402287034900-How-do-I-download-files-
To get a local copy of the csv files, select File at the top and then click "open"
this will open a new tab, select the files
'''
#views the first 5 rows of the afr file to confirm code ran as expected
amr.head()#removed data to comply with AOU data user policies. Shows the data extracted in tabular form
#includes all variables from the rows of the matrix tables, ancestry tables, and VAT table

In [ ]:
#copying the AMR file for download to continue downstream analysis, ficitious file
!gsutil cp gs://fc-secure-123-456-789/2026_B2_CT_variants_amr.csv .